In [ ]:
# ============================================================
# Notebook 4 Cell 1: Setup and parse experiment logs
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, re, glob, json, csv
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = '/content/drive/MyDrive/cs4782_matcher'
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(os.path.join(RESULTS_DIR, 'plots'),  exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, 'tables'), exist_ok=True)

# Download FSS-1000 test split if missing (needed for class names, ~3 KB download)
SPLITS_PATH = '/content/datasets/FSS-1000/splits/test.txt'
if not os.path.exists(SPLITS_PATH):
    os.makedirs('/content/datasets/FSS-1000/splits', exist_ok=True)
    !wget -q https://raw.githubusercontent.com/juhongm999/hsnet/main/data/splits/fss/test.txt \
        -O {SPLITS_PATH}
    print(f"Downloaded: {SPLITS_PATH}")

def parse_log(log_path):
    """Parse a Matcher log.txt. Returns None if file missing or empty."""
    if log_path is None or not os.path.exists(log_path):
        return None
    with open(log_path, 'r') as f:
        text = f.read()
    if not text.strip():
        return None

    # Final summary line: "Fold 0 mIoU: XX.XX   FB-IoU: YY.YY"
    fold_match = re.search(
        r'Fold\s+\d+\s+mIoU:\s+([\d.]+)\s+FB-IoU:\s+([\d.]+)', text)
    if not fold_match:
        return None

    final_miou   = float(fold_match.group(1))
    final_fb_iou = float(fold_match.group(2))

    # Per-class IoUs from the last [Batch: ...] line
    # Note: FSS-1000 fold 0 has 20 test classes (120 episodes each = 2400 total)
    batch_lines = re.findall(r'\[Batch:.*', text)
    per_class = []
    if batch_lines:
        last_line = batch_lines[-1]
        per_class = [float(m.group(1))
                     for m in re.finditer(r'\d+:\s+([\d.]+)', last_line)]

    return {
        'final_miou': final_miou,
        'final_fb_iou': final_fb_iou,
        'per_class_iou': per_class,
        'n_classes': len(per_class),
    }

# Parse both experiment logs
main_log_path = os.path.join(RESULTS_DIR, 'logs', 'main_with_ilm.log')
abl_log_path  = os.path.join(RESULTS_DIR, 'logs', 'ablation_forward_only.log')

main_result = parse_log(main_log_path)
abl_result  = parse_log(abl_log_path)

# Get class names for fold 0 (first 20 classes after sorting test.txt)
with open(SPLITS_PATH) as f:
    test_classes_all = sorted([l.strip() for l in f if l.strip()])
fold0_classes = test_classes_all[:20]

print("="*60)
print("Main experiment (Matcher, full bidirectional matching):")
if main_result:
    print(f"  Final mIoU:   {main_result['final_miou']:.2f}  (paper: 87.0)")
    print(f"  Final FB-IoU: {main_result['final_fb_iou']:.2f}")
    print(f"  # classes:    {main_result['n_classes']}")
else:
    print("  NOT AVAILABLE")

print("\nAblation experiment (forward-only matching, reverse disabled):")
if abl_result:
    print(f"  Final mIoU:   {abl_result['final_miou']:.2f}  (paper: 81.1)")
    print(f"  Final FB-IoU: {abl_result['final_fb_iou']:.2f}")
    print(f"  # classes:    {abl_result['n_classes']}")
else:
    print("  NOT AVAILABLE")
print("="*60)

print(f"\nFold 0 class names: {fold0_classes}")

In [ ]:
# ============================================================
# Notebook 4 Cell 2: Save summary tables
# ============================================================
PAPER_MAIN_FOLD0 = 87.0   # Paper Table 1: reported as mean of all folds, fold 0 expected similar
PAPER_ABL        = 81.1   # Paper Table 4b: forward-only
PAPER_DELTA      = PAPER_MAIN_FOLD0 - PAPER_ABL  # 5.9

# --- JSON summary ---
summary = {
    'benchmark': 'FSS-1000 (fold 0: 20 test classes, 120 episodes per class)',
    'hardware':  'NVIDIA L4 GPU, Google Colab Pro',
    'main_bidirectional': {
        'mIoU_ours':   main_result['final_miou']   if main_result else None,
        'FB_IoU_ours': main_result['final_fb_iou'] if main_result else None,
        'mIoU_paper':  PAPER_MAIN_FOLD0,
        'paper_reference': 'Matcher paper Table 1 (all folds mean)',
        'n_classes':   main_result['n_classes']    if main_result else 0,
    },
    'ablation_forward_only': {
        'mIoU_ours':   abl_result['final_miou']   if abl_result else None,
        'FB_IoU_ours': abl_result['final_fb_iou'] if abl_result else None,
        'mIoU_paper':  PAPER_ABL,
        'paper_reference': 'Matcher paper Table 4b ("forward" row)',
        'n_classes':   abl_result['n_classes']    if abl_result else 0,
    },
}
if main_result and abl_result:
    summary['bidirectional_contribution'] = {
        'delta_ours':  main_result['final_miou'] - abl_result['final_miou'],
        'delta_paper': PAPER_DELTA,
    }

with open(os.path.join(RESULTS_DIR, 'tables', 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

# --- Markdown summary ---
md  = "# FSS-1000 One-Shot Segmentation - Reproduction Results\n\n"
md += "## Setup\n"
md += "- **Benchmark**: FSS-1000, fold 0 (20 test classes, 120 episodes per class, 2400 total)\n"
md += "- **Hardware**: NVIDIA L4 GPU (Google Colab Pro)\n\n"
md += "## Results\n\n"
md += "| Setting                                 | mIoU (Ours) | mIoU (Paper) | FB-IoU (Ours) |\n"
md += "|------------------------------------------|-------------|--------------|----------------|\n"
if main_result:
    md += f"| Matcher (full, bidirectional matching)   | **{main_result['final_miou']:.2f}**     | {PAPER_MAIN_FOLD0}         | {main_result['final_fb_iou']:.2f}          |\n"
if abl_result:
    md += f"| Matcher w/o bidirectional (forward-only) | **{abl_result['final_miou']:.2f}**     | {PAPER_ABL}         | {abl_result['final_fb_iou']:.2f}          |\n"
if main_result and abl_result:
    delta = main_result['final_miou'] - abl_result['final_miou']
    md += f"| **Bidirectional matching contribution**  | **+{delta:.2f}** | **+{PAPER_DELTA:.1f}**     | -              |\n"
md += "\n*Note: Paper's 87.0 is the mean across all FSS-1000 folds. "
md += "We evaluated only fold 0 due to compute constraints; per-fold variance in the paper is ~1 mIoU.*\n"

with open(os.path.join(RESULTS_DIR, 'tables', 'summary.md'), 'w') as f:
    f.write(md)

# --- Per-class CSV ---
if main_result and main_result['per_class_iou']:
    ious_main = main_result['per_class_iou']
    n_classes = len(ious_main)
    class_names = fold0_classes[:n_classes]

    csv_path = os.path.join(RESULTS_DIR, 'tables', 'per_class_iou.csv')
    with open(csv_path, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['class_index', 'class_name',
                    'mIoU_bidirectional', 'mIoU_forward_only', 'delta'])
        for i, (name, iou_m) in enumerate(zip(class_names, ious_main)):
            iou_a = (abl_result['per_class_iou'][i]
                     if abl_result and i < len(abl_result['per_class_iou'])
                     else None)
            delta = (iou_m - iou_a) if iou_a is not None else None
            w.writerow([i, name, f'{iou_m:.2f}',
                        f'{iou_a:.2f}' if iou_a is not None else '',
                        f'{delta:+.2f}' if delta is not None else ''])
    print(f"Saved per-class CSV: {csv_path}")

print("\n=== Markdown summary ===\n")
print(md)

In [ ]:
# ============================================================
# Notebook 4 Cell 3: Bar chart comparing ours vs paper
# ============================================================
if not main_result:
    print("SKIPPED: main experiment data not available.")
else:
    have_ablation = abl_result is not None

    fig, ax = plt.subplots(figsize=(9, 5.5))

    if have_ablation:
        settings = ['Matcher (full,\nbidirectional)', 'Matcher w/o\nreverse matching']
        ours   = [main_result['final_miou'], abl_result['final_miou']]
        paper  = [87.0, 81.1]
    else:
        settings = ['Matcher (full)']
        ours   = [main_result['final_miou']]
        paper  = [87.0]

    x = np.arange(len(settings))
    width = 0.35

    bars_ours  = ax.bar(x - width/2, ours,  width, label='Ours (fold 0)',
                        color='#2E86AB', edgecolor='black')
    bars_paper = ax.bar(x + width/2, paper, width, label='Paper',
                        color='#A23B72', edgecolor='black')

    for bar, val in zip(bars_ours, ours):
        ax.text(bar.get_x() + bar.get_width()/2, val + 1.2,
                f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')
    for bar, val in zip(bars_paper, paper):
        ax.text(bar.get_x() + bar.get_width()/2, val + 1.2,
                f'{val:.1f}', ha='center', fontsize=11)

    ax.set_ylabel('mIoU (%)', fontsize=13)
    ax.set_title('FSS-1000 One-Shot Semantic Segmentation:\n'
                 'Our Reproduction (fold 0) vs. Paper (Table 1 & Table 4b)',
                 fontsize=13, pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels(settings, fontsize=12)
    ax.legend(fontsize=11, loc='upper right')
    ax.set_ylim(0, 105)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

    plt.tight_layout()
    out_path = os.path.join(RESULTS_DIR, 'plots', 'miou_comparison.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {out_path}")

In [ ]:
# ============================================================
# Notebook 4 Cell 4: Per-class IoU distribution
# ============================================================
if not main_result or not main_result['per_class_iou']:
    print("SKIPPED: per-class data not available.")
else:
    ious_main = main_result['per_class_iou']
    have_ablation = abl_result is not None and abl_result['per_class_iou']

    fig, ax = plt.subplots(figsize=(11, 5.5))

    if have_ablation:
        ious_abl = abl_result['per_class_iou']
        bins = np.linspace(0, 100, 21)  # bins of width 5
        ax.hist(ious_main, bins=bins, alpha=0.7, color='#2E86AB',
                edgecolor='black', label=f'Bidirectional (mean {np.mean(ious_main):.1f})')
        ax.hist(ious_abl,  bins=bins, alpha=0.7, color='#E63946',
                edgecolor='black', label=f'Forward-only (mean {np.mean(ious_abl):.1f})')
        ax.set_title('Per-class mIoU Distribution: Bidirectional vs Forward-only\n'
                     f'(FSS-1000 fold 0, {len(ious_main)} classes)',
                     fontsize=13, pad=10)
    else:
        ax.hist(ious_main, bins=20, color='#2E86AB', edgecolor='black', alpha=0.8)
        ax.axvline(np.mean(ious_main), color='red', linestyle='--', linewidth=2,
                   label=f'Mean = {np.mean(ious_main):.2f}')
        ax.set_title(f'Per-class mIoU Distribution (FSS-1000 fold 0, {len(ious_main)} classes)',
                     fontsize=13, pad=10)

    ax.set_xlabel('Per-class mIoU (%)', fontsize=12)
    ax.set_ylabel('Number of classes', fontsize=12)
    ax.legend(fontsize=11)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.set_xlim(0, 100)

    plt.tight_layout()
    out_path = os.path.join(RESULTS_DIR, 'plots', 'per_class_iou_histogram.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {out_path}")

    # Print stats
    print(f"\nMain experiment (bidirectional):")
    print(f"  Mean:   {np.mean(ious_main):.2f}")
    print(f"  Median: {np.median(ious_main):.2f}")
    print(f"  Min:    {min(ious_main):.2f}")
    print(f"  Max:    {max(ious_main):.2f}")
    print(f"  Std:    {np.std(ious_main):.2f}")
    if have_ablation:
        print(f"\nAblation (forward-only):")
        print(f"  Mean:   {np.mean(ious_abl):.2f}")
        print(f"  Median: {np.median(ious_abl):.2f}")
        print(f"  Min:    {min(ious_abl):.2f}")
        print(f"  Max:    {max(ious_abl):.2f}")
        print(f"  Std:    {np.std(ious_abl):.2f}")

In [ ]:
# ============================================================
# Notebook 4 Cell 5: Per-class comparison between main and ablation
# ============================================================
if not main_result or not main_result['per_class_iou']:
    print("SKIPPED: main experiment data not available.")
else:
    ious_main = main_result['per_class_iou']
    n_classes = len(ious_main)
    class_names = fold0_classes[:n_classes]
    have_ablation = abl_result is not None and abl_result['per_class_iou']

    if have_ablation:
        ious_abl = abl_result['per_class_iou']

        # Sort by main experiment's IoU for easier reading
        order = np.argsort(ious_main)[::-1]  # descending
        sorted_names = [class_names[i] for i in order]
        sorted_main  = [ious_main[i]   for i in order]
        sorted_abl   = [ious_abl[i]    for i in order]

        fig, ax = plt.subplots(figsize=(12, 8))
        y = np.arange(n_classes)
        height = 0.4

        ax.barh(y - height/2, sorted_main, height, color='#2E86AB',
                edgecolor='black', label='Bidirectional (full)')
        ax.barh(y + height/2, sorted_abl,  height, color='#E63946',
                edgecolor='black', label='Forward-only (ablation)')

        ax.set_yticks(y)
        ax.set_yticklabels(sorted_names, fontsize=10)
        ax.set_xlabel('mIoU (%)', fontsize=12)
        ax.set_title(f'Per-class mIoU: Bidirectional vs. Forward-only\n'
                     f'(FSS-1000 fold 0, sorted by bidirectional mIoU)',
                     fontsize=13, pad=10)
        ax.legend(fontsize=11, loc='lower right')
        ax.set_xlim(0, 105)
        ax.grid(axis='x', linestyle='--', alpha=0.5)
        ax.invert_yaxis()  # best class at top

        plt.tight_layout()
        out_path = os.path.join(RESULTS_DIR, 'plots', 'per_class_comparison.png')
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved: {out_path}")

        # Print the top classes where bidirectional helps most
        deltas = [(class_names[i], ious_main[i] - ious_abl[i])
                  for i in range(n_classes)]
        deltas_sorted = sorted(deltas, key=lambda x: x[1], reverse=True)
        print("\n=== Classes where bidirectional matching helps most ===")
        for name, d in deltas_sorted[:5]:
            print(f"  {name:30s}  +{d:.2f} mIoU")
        print("\n=== Classes where bidirectional matching barely helps or hurts ===")
        for name, d in deltas_sorted[-5:]:
            sign = '+' if d >= 0 else ''
            print(f"  {name:30s}  {sign}{d:.2f} mIoU")

    else:
        # Ablation not available - just show main experiment as a sorted bar chart
        order = np.argsort(ious_main)[::-1]
        sorted_names = [class_names[i] for i in order]
        sorted_vals  = [ious_main[i]   for i in order]

        fig, ax = plt.subplots(figsize=(11, 7))
        ax.barh(range(n_classes), sorted_vals, color='#2E86AB', edgecolor='black')
        ax.set_yticks(range(n_classes))
        ax.set_yticklabels(sorted_names, fontsize=10)
        ax.set_xlabel('mIoU (%)', fontsize=12)
        ax.set_title(f'Per-class mIoU (FSS-1000 fold 0, {n_classes} classes)',
                     fontsize=13, pad=10)
        ax.set_xlim(0, 105)
        ax.grid(axis='x', linestyle='--', alpha=0.5)
        ax.invert_yaxis()
        for i, v in enumerate(sorted_vals):
            ax.text(v + 1, i, f'{v:.1f}', va='center', fontsize=9)
        plt.tight_layout()
        out_path = os.path.join(RESULTS_DIR, 'plots', 'per_class_bar_chart.png')
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved: {out_path}")

# ============================================================
# Final listing of everything in results/
# ============================================================
print("\n" + "="*60)
print("ALL FINAL OUTPUTS SAVED TO results/")
print("="*60)
!ls -la /content/drive/MyDrive/cs4782_matcher/results/
print("\n--- plots/ ---")
!ls -la /content/drive/MyDrive/cs4782_matcher/results/plots/
print("\n--- tables/ ---")
!ls -la /content/drive/MyDrive/cs4782_matcher/results/tables/
print("\n--- logs/ ---")
!ls -la /content/drive/MyDrive/cs4782_matcher/results/logs/